# SAM Polygon Point Extraction

Runs SAM (ViT-B) on PCB component bounding boxes to extract polygon boundaries and export coordinates to CSV and Excel.

In [ ]:
# Dependencies
!pip install -q opencv-python numpy pandas openpyxl matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
# Model checkpoint
import urllib.request

SAM_CHECKPOINT = Path('sam_vit_b.pth')
if not SAM_CHECKPOINT.exists():
    print('Downloading sam_vit_b.pth...')
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth', str(SAM_CHECKPOINT))

sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CHECKPOINT))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)

In [ ]:
def extract_polygon_points(mask: np.ndarray, min_area: float = 15.0):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return []
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area:
        return []
    epsilon = 0.005 * cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, epsilon, True)
    return [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]

In [ ]:
# Run extraction
image_path = Path('dataset_split/train/images/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.jpg')
labels_path = Path('dataset_split/train/labels/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.txt')
output_dir = Path('sam_points_output')
output_dir.mkdir(parents=True, exist_ok=True)

img = cv2.imread(str(image_path))
h, w = img.shape[:2]
predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

CLASS_MAP = {0: 'Cap1', 1: 'Cap2', 2: 'Cap3', 3: 'Cap4', 4: 'MOSFET', 5: 'Mov', 6: 'Resistor', 7: 'Transformer'}
PREFIX_MAP = {'Cap1': 'C', 'Cap2': 'C', 'Cap3': 'C', 'Cap4': 'C', 'MOSFET': 'Q', 'Mov': 'D', 'Resistor': 'R', 'Transformer': 'T'}

boxes = []
with open(labels_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            cid = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = int(round((xc - bw / 2.0) * w))
            y1 = int(round((yc - bh / 2.0) * h))
            x2 = int(round((xc + bw / 2.0) * w))
            y2 = int(round((yc + bh / 2.0) * h))
            boxes.append(([x1, y1, x2, y2], cid))

records = []
shapes = []
ref_counts = {}
vis_img = img.copy()

for idx, (b, cid) in enumerate(boxes, 1):
    masks, scores, _ = predictor.predict(box=np.array(b)[None, :], multimask_output=False)
    conf = float(scores[0])
    pts = extract_polygon_points(masks[0])
    if not pts:
        pts = [[float(b[0]), float(b[1])], [float(b[2]), float(b[1])], [float(b[2]), float(b[3])], [float(b[0]), float(b[3])]]
    
    class_name = CLASS_MAP.get(cid, f'Class_{cid}')
    prefix = PREFIX_MAP.get(class_name, 'U')
    ref_counts[prefix] = ref_counts.get(prefix, 0) + 1
    ref_des = f'{prefix}{ref_counts[prefix]}'
    
    records.append({
        'image_name': image_path.name,
        'instance_id': idx,
        'ref_des': ref_des,
        'class_name': class_name,
        'box_x1': b[0], 'box_y1': b[1], 'box_x2': b[2], 'box_y2': b[3],
        'confidence': round(conf, 3),
        'num_polygon_points': len(pts),
        'polygon_points_compact': '; '.join([f'({p[0]},{p[1]})' for p in pts]),
        'polygon_points_json': json.dumps(pts)
    })
    shapes.append({
        'label': f'{ref_des}: {class_name}',
        'points': pts,
        'shape_type': 'polygon'
    })
    cnt = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(vis_img, [cnt], True, (0, 255, 255), 2)

df = pd.DataFrame(records)
df.to_excel(output_dir / 'sam_output_points.xlsx', index=False)
df.to_csv(output_dir / 'sam_output_points.csv', index=False)

with open(output_dir / f'{image_path.stem}.json', 'w') as f:
    json.dump({'version': '5.5.0', 'flags': {}, 'shapes': shapes, 'imagePath': image_path.name, 'imageData': None, 'imageHeight': h, 'imageWidth': w}, f, indent=2)

print(f'Processed {len(records)} components.')
df.head()

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()